# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ARS-0/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

*Clones my repo on Colab, then rebuilds the exact Week-5 data, label and feature set so the audit tests the model I actually reported.*

In [1]:
import os, sys, subprocess, json, warnings
import numpy as np
import pandas as pd
import sklearn
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ARS-0/FlyRank-ML-Internship"
REPO_DIR = "FlyRank-ML-Internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"
os.makedirs("work/outputs", exist_ok=True)

SEED = 42
np.random.seed(SEED)
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Rows, columns:", df.shape, "| scikit-learn", sklearn.__version__, "| seed", SEED)

Rows, columns: (30000, 44) | scikit-learn 1.8.0 | seed 42


In [2]:
# Same setup as my Week-5 notebook, so the audit tests the exact model I reported.
#   OUTCOME window = last 30 days. FEATURE window = days 31-90 back (90-day total minus last 30 days).
df["pre_impr"] = df["impressions_90d"] - df["impressions_last_30d"]
df["pre_clk"]  = df["clicks_90d"] - df["clicks_last_30d"]
e = df[(df["pre_impr"] >= 500) & (df["impressions_last_30d"] >= 500) & (df["avg_position"] > 0)].reset_index(drop=True)

bench = e.groupby("position_tier").apply(lambda x: x["clicks_last_30d"].sum() / x["impressions_last_30d"].sum())
e["y"] = (e["clicks_last_30d"] <= 0.5 * e["position_tier"].map(bench) * e["impressions_last_30d"]).astype(int)

pre_rate = e.groupby("position_tier").apply(lambda x: x["pre_clk"].sum() / x["pre_impr"].sum())
e["rule_score"] = (e["position_tier"].map(pre_rate) * e["pre_impr"] - e["pre_clk"]).clip(lower=0)   # W04 rule, feature window

e["log_pre_impr"]    = np.log1p(e["pre_impr"]);  e["log_pre_clk"] = np.log1p(e["pre_clk"])
e["pre_ctr_pct"]     = e["pre_clk"] / e["pre_impr"] * 100
e["log_prev30_impr"] = np.log1p(e["impressions_prev_30d"])
e["prev30_ctr_pct"]  = np.where(e["impressions_prev_30d"] > 0, e["clicks_prev_30d"] / e["impressions_prev_30d"].clip(lower=1) * 100, 0)
e["has_word_count"]  = e["word_count"].notna().astype(int); e["word_count_f"] = e["word_count"].fillna(e["word_count"].median())
e["has_volume"]      = e["search_volume"].notna().astype(int); e["log_volume"] = np.log1p(e["search_volume"].fillna(0))
e["competition_f"]   = e["competition"].fillna(0); e["cpc_f"] = e["cpc"].fillna(0); e["log_age"] = np.log1p(e["content_age_days"])

STATIC  = ["avg_position", "log_volume", "competition_f", "cpc_f", "word_count_f", "has_word_count", "has_volume", "log_age"]
WINDOW  = ["log_pre_impr", "log_pre_clk", "pre_ctr_pct", "log_prev30_impr", "prev30_ctr_pct"]
CAT     = pd.get_dummies(e[["content_type", "main_intent", "position_tier"]], dtype=int)
X_honest = pd.concat([e[WINDOW + STATIC], CAT], axis=1)               # = my Week-5 feature set

# A deliberately NAIVE feature set: the 90-day columns, which CONTAIN the outcome window
e["log_imp90"] = np.log1p(e["impressions_90d"]); e["log_clk90"] = np.log1p(e["clicks_90d"])
X_naive = pd.concat([e[["ctr", "log_imp90", "log_clk90"] + STATIC], CAT], axis=1)

y = e["y"].values; groups = e["client_id"].values
print(f"Eligible pages: {len(e):,} | clients: {e['client_id'].nunique()} | base rate: {y.mean():.3f}")
print("Honest features:", X_honest.shape[1], "| naive features:", X_naive.shape[1])

Eligible pages: 9,102 | clients: 28 | base rate: 0.361
Honest features: 25 | naive features: 23


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

I read FlyRank's March 2026 paper (*The State of AI-Driven SEO in Numbers*). It leads with plain portfolio comparisons, labels its ML pages "exploratory," and says outright that its feature-importance and growth models are descriptive, not causal. It also flags its own weak spots (the `361+` freshness bucket has 1 declining page, so it refuses to headline it). I want to review it the way I'd want my own work reviewed. The two questions below are about *how far a number can be read*, not about whether anyone got it wrong.

### Finding 1 — ML appendix, "What Predicts Growth?" (logistic regression, 71% holdout accuracy)

**Where does the label come from?** The paper defines growing vs declining from the change in impressions between the last 30 days and the 30 days before. The features include impressions, clicks and days visible, and the paper says recent impressions are among the strongest positive signals. If those features were measured over a window that includes the same 30 days the label was built from, a good score would be expected, because the model would partly be reading the answer. **My question:** were the features measured strictly *before* the 30-day window that defines the label?

**Does the validation design carry the claim?** The paper says "holdout" but not how the holdout was drawn. With 57 brands, a random holdout lets pages from the same brand sit on both sides, and a grouped-by-brand holdout asks the harder question. It also doesn't give the base rate. In my own starter file, 54.2% of pages are labelled "down," so a model that always says "down" is already right about that often (the cell below prints this). **My question:** how was the holdout drawn (random or by brand), and what is the accuracy of always guessing the majority class? Read against that, 71% could be a large or a small improvement. One small thing I'd also ask: the paper's trend threshold is ±10%, while my data dictionary uses ±20%, so which one produced the label?

### Finding 4 — "The Freshness Multiplier" (365+ day pages refreshed in the last 30 days: health 10.7 → 34.5, impressions 71 → 4,039)

**Where does the outcome come from?** Health score is built partly from impressions (30 points), position, CTR and scroll. So "3.2x health" and "57x impressions" are partly the same measurement counted twice. **My question:** what does the comparison look like using raw impressions and clicks only?

**Does the design carry the claim?** The paper shows a refreshed group next to an unrefreshed one, but not the same pages before and after. If refreshed pages were picked because they already had demand, the gap could come from *why* they were chosen. The averages (71 vs 4,039) are also from a heavy-tailed distribution, where a few big pages can pull a mean far up, and the group sizes aren't shown. **My question:** is it the same pages before and after, or two different groups at one moment, and what are the group sizes and medians? The paper already does this kind of care for the `361+` bucket, so I'm only asking for the same care here.

### What I could check myself

The cell below compares the paper's Finding 3 (weighted CTR by position tier) with my starter file. The tiers come out in the same order, and the values are close for four of the five tiers (the top-3 tier is the furthest apart: 0.49% vs 0.42%, on a smaller sample). That is a small independent check that I'm reading the data the same way the paper did.

In [3]:
# One thing in the paper I CAN check on my own data: Finding #3, weighted CTR by position tier.
# Paper (p.8, weighted = total clicks / total impressions, in %):
paper = {"top_3": 0.423, "page_1": 0.339, "striking": 0.325, "page_3_5": 0.163, "deep": 0.050}
order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
mine = df[df["position_tier"].isin(order)].groupby("position_tier").apply(lambda x: x["clicks_90d"].sum() / x["impressions_90d"].sum() * 100)
cmp_tbl = pd.DataFrame({"paper_weighted_ctr_pct": pd.Series(paper), "my_starter_csv_pct": mine.loc[order].round(3),
                        "pages_in_my_tier": df["position_tier"].value_counts().loc[order]})
print(cmp_tbl.to_string())
print("\nSame order from best tier to worst in both:", list(cmp_tbl["paper_weighted_ctr_pct"].sort_values(ascending=False).index) ==
      list(cmp_tbl["my_starter_csv_pct"].sort_values(ascending=False).index))

# Small facts behind my two questions, from the starter CSV:
print("\nStarter CSV: share of pages labelled trend_direction == 'down':", round((df["trend_direction"] == "down").mean(), 3),
      "-> a model that always says 'down' would already be right about that often.")

          paper_weighted_ctr_pct  my_starter_csv_pct  pages_in_my_tier
top_3                      0.423               0.488              2321
page_1                     0.339               0.350             11814
striking                   0.325               0.347              7304
page_3_5                   0.163               0.155              7242
deep                       0.050               0.041              1319

Same order from best tier to worst in both: True

Starter CSV: share of pages labelled trend_direction == 'down': 0.542 -> a model that always says 'down' would already be right about that often.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

This is my Week-5 model, tested again under four setups. The **before** number is what I would have reported with a careless design: the 90-day columns (which contain the outcome window) and a random split. The **after** number removes both problems, in two steps, so I can see how much each one was inflating the score.

- **A (before):** 90-day features + random split
- **B:** window-only features + random split (removes the overlap)
- **C (after):** window-only features + split by client (removes client memorisation). This is my Week-5 design.
- **D:** 90-day features + split by client (shows what the overlap alone is worth)

Base rate is printed next to every score. The data is a single snapshot, so a time-aware split isn't possible; the outcome window is already the later one.

In [4]:
from sklearn.model_selection import KFold, GroupKFold, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores), kind="stable")[:k]
    return np.asarray(labels)[order].mean()

def make_models():
    return {"Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.5)),
            "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=20, max_features="sqrt", random_state=SEED, n_jobs=-1)}

def evaluate(Xmat, splits):
    rows = []
    for tr, te in splits:
        r = {"base_rate": y[te].mean(), "Rule baseline p@50": precision_at_k(e["rule_score"].values[te], y[te], 50)}
        for name, m in make_models().items():
            p = m.fit(Xmat.iloc[tr], y[tr]).predict_proba(Xmat.iloc[te])[:, 1]
            r[f"{name} p@50"] = precision_at_k(p, y[te], 50)
            r[f"{name} p@200"] = precision_at_k(p, y[te], 200)
            r[f"{name} AUC"] = roc_auc_score(y[te], p)
        rows.append(r)
    return pd.DataFrame(rows).mean()

random_splits  = list(KFold(5, shuffle=True, random_state=SEED).split(X_honest))
grouped_splits = list(GroupKFold(5).split(X_honest, y, groups))
for tr, te in grouped_splits:
    assert not set(groups[tr]) & set(groups[te])          # no client on both sides

setups = {
    "A  BEFORE: 90-day features + random split":        (X_naive,  random_splits),
    "B  window-only features + random split":           (X_honest, random_splits),
    "C  AFTER:  window-only features + split by client": (X_honest, grouped_splits),
    "D  90-day features + split by client":              (X_naive,  grouped_splits),
}
res = pd.DataFrame({k: evaluate(*v) for k, v in setups.items()}).T
cols = ["base_rate", "Rule baseline p@50", "Logistic Regression p@50", "Random Forest p@50", "Random Forest p@200", "Random Forest AUC"]
print(res[cols].round(3).to_string())
print("\nRead it as: A -> B removes the outcome-window columns; B -> C removes client memorisation. A -> C is the full correction.")

                                                   base_rate  Rule baseline p@50  Logistic Regression p@50  Random Forest p@50  Random Forest p@200  Random Forest AUC
A  BEFORE: 90-day features + random split              0.361               0.724                     0.984               0.996                0.952              0.912
B  window-only features + random split                 0.361               0.724                     0.880               0.928                0.854              0.849
C  AFTER:  window-only features + split by client      0.344               0.704                     0.800               0.816                0.752              0.834
D  90-day features + split by client                   0.344               0.704                     0.964               0.972                0.895              0.900

Read it as: A -> B removes the outcome-window columns; B -> C removes client memorisation. A -> C is the full correction.


In [5]:
# How much does the honest number move if I hold out DIFFERENT clients? 20 random client-holdouts (25% of clients each).
gss = list(GroupShuffleSplit(n_splits=20, test_size=0.25, random_state=SEED).split(X_honest, y, groups))
rows = []
for tr, te in gss:
    r = {"test_pages": len(te), "base": y[te].mean(), "rule": precision_at_k(e["rule_score"].values[te], y[te], 50)}
    for name, m in make_models().items():
        r[name] = precision_at_k(m.fit(X_honest.iloc[tr], y[tr]).predict_proba(X_honest.iloc[te])[:, 1], y[te], 50)
    rows.append(r)
rep = pd.DataFrame(rows)
print("Precision@50 over 20 client-holdouts (mean / std / min / max):")
print(rep[["base", "rule", "Logistic Regression", "Random Forest"]].agg(["mean", "std", "min", "max"]).round(3).to_string())
for name in ["Logistic Regression", "Random Forest"]:
    diff = rep[name] - rep["rule"]
    print(f"{name}: beats the rule in {(diff > 0).sum()} of 20 holdouts | mean gain {diff.mean():+.3f} (std {diff.std():.3f})")

# And without the single biggest client (it holds about a third of all pages)?
big = e["client_id"].value_counts().index[0]
idx = np.where(e["client_id"] != big)[0]
sub = [(idx[a], idx[b]) for a, b in GroupKFold(5).split(X_honest.iloc[idx], y[idx], groups[idx])]
r_nb = evaluate(X_honest, sub)
print(f"\nWithout the biggest client ({(e['client_id']==big).mean():.0%} of pages):")
print(r_nb[["base_rate", "Rule baseline p@50", "Logistic Regression p@50", "Random Forest p@50"]].round(3).to_string())

Precision@50 over 20 client-holdouts (mean / std / min / max):
       base   rule  Logistic Regression  Random Forest
mean  0.330  0.680                0.794          0.846
std   0.066  0.113                0.078          0.081
min   0.240  0.400                0.680          0.600
max   0.458  0.840                0.960          0.940
Logistic Regression: beats the rule in 18 of 20 holdouts | mean gain +0.114 (std 0.095)
Random Forest: beats the rule in 19 of 20 holdouts | mean gain +0.166 (std 0.091)



Without the biggest client (35% of pages):
base_rate                   0.315
Rule baseline p@50          0.660
Logistic Regression p@50    0.752
Random Forest p@50          0.808


### What changed, with the base rate in view

Base rate is about 0.34-0.36, and the rule baseline gets 0.70-0.72 at precision@50.

| Setup | Logistic Regression p@50 | Random Forest p@50 |
|---|---|---|
| A: 90-day features, random split (before) | 0.984 | 0.996 |
| B: window-only, random split | 0.880 | 0.928 |
| C: window-only, split by client (after) | 0.800 | 0.816 |
| D: 90-day features, split by client | 0.964 | 0.972 |

- **The careless version looks almost perfect (0.996) and the honest one is 0.816.** That is about 18 points of the first number that was not skill.
- **Both corrections matter.** Moving from A to B (removing the overlap) costs about 7 points for Random Forest. Moving from B to C (grouping by client) costs about 11 more. Row D shows the overlap alone is worth about 16 points even when clients are held out.
- **The honest number holds up when I swap which clients are held out.** Over 20 random client holdouts, Logistic Regression beat my rule at precision@50 in 18 of 20, and Random Forest in 19 of 20. The mean gain was +0.11 (Logistic Regression) and +0.17 (Random Forest), with a spread of about 0.09-0.10, so a single holdout can land well above or below that. Across holdouts, Random Forest ranged from 0.60 to 0.94 at precision@50, and the rule from 0.40 to 0.84.
- **The biggest client is not carrying it.** Without the client that holds 35% of the pages, the rule gets 0.66, Logistic Regression 0.75 and Random Forest 0.81. The gains stay positive (about +0.09 for Logistic Regression and +0.15 for Random Forest).

My reading: the model's advantage over my rule is directional and fairly consistent across client holdouts, and the size varies a lot by which clients are held out.

## 3. Leakage audit

*The same hunt from Week 3, on my final feature set — plus real failure examples.*

I went through the same checklist from the leakage skill. First an audit of every feature by the window it comes from, then a test to see if my harness can catch a leak at all (I planted one on purpose), then retraining without the suspects, then a look at who is in my population. Last come three real wrong cases in each direction.

In [6]:
# ---- 1) Feature-by-feature audit ----
def window_of(col):
    if col in WINDOW: return "feature window (days 31-90 back)"
    if col == "avg_position" or col.startswith("position_tier_"): return "90 days - OVERLAPS outcome window"
    return "static page fact (not a performance window)"
audit = pd.DataFrame({"feature": X_honest.columns})
audit["window"] = audit["feature"].map(window_of)
print(audit.groupby("window")["feature"].agg(["count", lambda s: ", ".join(s[:6]) + (" ..." if len(s) > 6 else "")]).to_string())

FORBIDDEN = ["trend_direction", "trend_pct", "is_declining_label", "health_score", "priority_score", "action_type",
             "ctr", "impressions_90d", "clicks_90d", "impressions_last_30d", "clicks_last_30d", "sessions_last_30d"]
bad = [c for c in X_honest.columns if c in FORBIDDEN or "last_30d" in c]
print("\nForbidden or outcome-window columns in the honest feature set:", bad)
assert not bad

# ---- 2) Does my test harness catch a leak? Plant one on purpose. ----
X_plant = X_honest.copy()
X_plant["PLANTED_outcome_ctr"] = e["clicks_last_30d"] / e["impressions_last_30d"] * 100
planted = evaluate(X_plant, random_splits)
print("\nPLANTED leak (the outcome window's own CTR as a feature):")
print(planted[["Logistic Regression p@50", "Logistic Regression AUC", "Random Forest p@50", "Random Forest AUC"]].round(3).to_string())
print("-> a score near 1.0 means the harness does detect leakage. I then removed it.")

# ---- 3) Train WITHOUT the suspects ----
def drop(cols): return X_honest.drop(columns=[c for c in X_honest.columns if c in cols])
tests = {"all honest features (Week 5)": X_honest,
         "without the two CTR features": drop(["pre_ctr_pct", "prev30_ctr_pct"]),
         "without avg_position + position tiers (overlap columns)": X_honest.drop(columns=[c for c in X_honest.columns if c == "avg_position" or c.startswith("position_tier_")]),
         "without static page facts": X_honest[WINDOW + [c for c in X_honest.columns if c == "avg_position" or c.startswith("position_tier_")]]}
tbl = pd.DataFrame({k: evaluate(v, grouped_splits) for k, v in tests.items()}).T
print("\nSplit by client, same folds as above:")
print(tbl[["Logistic Regression p@50", "Random Forest p@50", "Random Forest AUC"]].round(3).to_string())

# ---- 4) Population check: my eligibility rule looks at the OUTCOME window ----
cand = df[(df["pre_impr"] >= 500) & (df["avg_position"] > 0)].copy()
cand["kept"] = cand["impressions_last_30d"] >= 500
cand["pre_ctr_pct"] = cand["pre_clk"] / cand["pre_impr"] * 100
print(f"\nPages visible enough BEFORE the outcome window: {len(cand):,}. Kept (still >= 500 impressions in the last 30 days): {cand['kept'].sum():,}. Dropped: {(~cand['kept']).mean():.0%}")
print(cand.groupby("kept").agg(pages=("pre_impr", "size"), median_pre_impressions=("pre_impr", "median"),
                               mean_pre_ctr_pct=("pre_ctr_pct", "mean"), median_pre_ctr_pct=("pre_ctr_pct", "median")).round(3).to_string())

# ---- 5) Label benchmark: pooled outcome-window CTR per tier uses all clients (a fixed yardstick, not a feature) ----
print("\nLabel benchmark (pooled last-30d CTR %, all eligible clients):", (bench * 100).round(3).to_dict())

                                             count                                                                                                                   <lambda_0>
window                                                                                                                                                                         
90 days - OVERLAPS outcome window                6  avg_position, position_tier_deep, position_tier_page_1, position_tier_page_3_5, position_tier_striking, position_tier_top_3
feature window (days 31-90 back)                 5                                                      log_pre_impr, log_pre_clk, pre_ctr_pct, log_prev30_impr, prev30_ctr_pct
static page fact (not a performance window)     14                                               log_volume, competition_f, cpc_f, word_count_f, has_word_count, has_volume ...

Forbidden or outcome-window columns in the honest feature set: []



PLANTED leak (the outcome window's own CTR as a feature):
Logistic Regression p@50    1.000
Logistic Regression AUC     1.000
Random Forest p@50          1.000
Random Forest AUC           0.998
-> a score near 1.0 means the harness does detect leakage. I then removed it.



Split by client, same folds as above:
                                                         Logistic Regression p@50  Random Forest p@50  Random Forest AUC
all honest features (Week 5)                                                0.800               0.816              0.834
without the two CTR features                                                0.800               0.796              0.807
without avg_position + position tiers (overlap columns)                     0.784               0.812              0.824
without static page facts                                                   0.788               0.812              0.831

Pages visible enough BEFORE the outcome window: 15,441. Kept (still >= 500 impressions in the last 30 days): 9,102. Dropped: 41%
       pages  median_pre_impressions  mean_pre_ctr_pct  median_pre_ctr_pct
kept                                                                      
False   6339                  1053.0             0.181               0.109
T

In [7]:
# ---- Real failure examples (out-of-fold: each page is scored by a model that never saw its client) ----
oof = np.zeros(len(e))
for tr, te in grouped_splits:
    oof[te] = make_models()["Random Forest"].fit(X_honest.iloc[tr], y[tr]).predict_proba(X_honest.iloc[te])[:, 1]
e["p_rf"] = oof
e["outcome_ctr_pct"] = e["clicks_last_30d"] / e["impressions_last_30d"] * 100
e["tier_bench_pct"] = e["position_tier"].map(bench) * 100

show = ["content_id", "position_tier", "pre_impr", "pre_clk", "pre_ctr_pct", "impressions_last_30d", "clicks_last_30d",
        "outcome_ctr_pct", "tier_bench_pct", "expected_clicks", "p_rf"]
e["expected_clicks"] = e["tier_bench_pct"] / 100 * e["impressions_last_30d"]

print("FALSE ALARMS: ranked as most likely under-capturing, but the page did fine in the outcome window")
print(e[e["y"] == 0].sort_values("p_rf", ascending=False).head(3)[show].round(3).to_string(index=False))
print("\nMISSES: pages that DID under-capture, but the model gave them the lowest scores")
print(e[e["y"] == 1].sort_values("p_rf").head(3)[show].round(3).to_string(index=False))

top200 = e.sort_values("p_rf", ascending=False).head(200)
print(f"\nTop-200 precision: {top200['y'].mean():.3f}  (base rate {y.mean():.3f}) | clients: {top200['client_id'].nunique()} | biggest client share: {top200['client_id'].value_counts(normalize=True).iloc[0]:.0%}")
print("\nHow often y=1 by position tier (all eligible pages):")
print(e.groupby("position_tier")["y"].agg(pages="size", share_y1="mean").round(3).to_string())

FALSE ALARMS: ranked as most likely under-capturing, but the page did fine in the outcome window
          content_id position_tier  pre_impr  pre_clk  pre_ctr_pct  impressions_last_30d  clicks_last_30d  outcome_ctr_pct  tier_bench_pct  expected_clicks  p_rf
content_08eb507fdfc2          deep      2203        1        0.045                  2019                1            0.050           0.024            0.490 0.892
content_983adc986531      page_3_5      2277        0        0.000                  1252                2            0.160           0.220            2.758 0.871
content_b1bc831deff1      striking     11081        0        0.000                   663                3            0.452           0.429            2.844 0.870

MISSES: pages that DID under-capture, but the model gave them the lowest scores
          content_id position_tier  pre_impr  pre_clk  pre_ctr_pct  impressions_last_30d  clicks_last_30d  outcome_ctr_pct  tier_bench_pct  expected_clicks  p_rf
content_2131

### Leakage audit

- **Features by window:** 5 come from the feature window (days 31-90 back), 14 are static page facts, and 6 (`avg_position` and the five position-tier columns) cover all 90 days, so they **overlap the outcome window**. I kept them because a position-adjusted CTR needs position, and I disclosed it in Week 5. Dropping them moves Random Forest from 0.816 to 0.812 and Logistic Regression from 0.800 to 0.784, so the result doesn't depend on them.
- **No forbidden columns:** no `trend_direction`, `trend_pct`, `is_declining_label`, no `_last_30d` column, and no 90-day totals. The product flags aren't in the file at all.
- **The planted leak was caught.** Adding the outcome window's own CTR pushes both models to about 1.0 (Logistic Regression precision@50 = 1.000, Random Forest AUC = 0.998). So my harness would show a leak if one were there. I removed it, and the honest number is 0.82.
- **The score isn't "too good."** The honest AUC is about 0.83 and precision@50 is about 0.80-0.82, nowhere near perfect.
- **A correction to Week 5.** There I said the model mostly learns that low CTR before means low CTR after. Dropping the two CTR columns barely changes the result (Random Forest 0.816 to 0.796), because past clicks and past impressions are still there and the model can rebuild CTR from them. The accurate statement is that the model leans on how many clicks a page earned relative to its visibility in the earlier window, however it is expressed.

### A selection choice I should have flagged more loudly

My eligibility rule keeps only pages with at least 500 impressions **in the outcome window** itself. That uses outcome information to choose who gets scored. Of 15,441 pages that were visible enough before the outcome window, 41% (6,339) were dropped for this reason, and the dropped pages had lower earlier CTR (median 0.109% vs 0.201%) and less volume (median 1,053 vs 5,408 earlier impressions). So the results describe **pages that stayed visible**. They say nothing about pages that lost their impressions, and I shouldn't suggest otherwise. The label's benchmark (pooled outcome-window CTR per tier) also uses all eligible clients. It is a fixed yardstick, not a feature, but it is outcome-window information too.

### Real failure examples

- **False alarms:** three pages the model was about 87-89% sure would under-capture, which did fine. `content_08eb507fdfc2` (deep tier) got 1 click against about 0.5 expected, so a single click flipped it. `content_983adc986531` and `content_b1bc831deff1` had 0 clicks in the earlier window, then recovered to about the tier benchmark (0.16% vs 0.22%, and 0.45% vs 0.43%). With so few clicks, a page's earlier record is weak evidence about the next month.
- **Misses:** three striking-distance pages the model scored lowest (2-3%) that did under-capture. Each had a decent earlier CTR (0.55-1.1%, above the 0.43% benchmark) and then dropped to 0.08-0.20% in the outcome window. A model reading only the earlier window can't see a sudden drop coming.
- **Who is in the top 200:** 87.5% are correct against a 36% base rate, but they come from 13 clients and 44% from one of them. The deep tier has only 105 eligible pages, 76% of them labelled `y = 1` (basically "zero clicks"), so I trust that tier least.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [8]:
# ---- Re-checking my own boldest Week-1 / Week-4 claim: "comparison articles underperform" ----
d = df[df["impressions_90d"] >= 100]
def pooled(z): return z["clicks_90d"].sum() / z["impressions_90d"].sum() * 100

print("Clients that have each content type (whole file):")
print(df.groupby("content_type")["client_id"].nunique().to_string())
comp_client = df.loc[df["content_type"] == "comparison article", "client_id"].unique()
print("\nClients with comparison articles:", len(comp_client), "->", list(comp_client))
print("Comparison articles at that client:", int((df["content_type"] == "comparison article").sum()),
      "| its keyword articles:", int(((df["client_id"] == comp_client[0]) & (df["content_type"] == "keyword article")).sum()))

rng = np.random.default_rng(SEED)
def boot_diff(a, b, n=1000):
    out = []
    for _ in range(n):
        out.append(pooled(a.sample(len(a), replace=True, random_state=int(rng.integers(1e9)))) -
                   pooled(b.sample(len(b), replace=True, random_state=int(rng.integers(1e9)))))
    return np.percentile(out, [2.5, 97.5])

rows = []
for tier in ["page_1", "striking", "page_3_5"]:
    comp = d[(d["position_tier"] == tier) & (d["content_type"] == "comparison article")]
    kw_all = d[(d["position_tier"] == tier) & (d["content_type"] == "keyword article")]
    kw_same = kw_all[kw_all["client_id"].isin(comp_client)]
    row = {"tier": tier, "comparison_n": len(comp), "comparison_ctr_pct": round(pooled(comp), 3),
           "kw_ALL_clients_n": len(kw_all), "kw_ALL_ctr_pct": round(pooled(kw_all), 3),
           "kw_SAME_client_n": len(kw_same), "kw_SAME_ctr_pct": round(pooled(kw_same), 3)}
    if len(kw_same) >= 30:
        lo, hi = boot_diff(kw_same, comp); row["same_client_gap_95%_range"] = f"{lo:.3f} to {hi:.3f}"
    else:
        row["same_client_gap_95%_range"] = "too few keyword pages (n < 30)"
    rows.append(row)
print("\nPooled CTR (%) comparison vs keyword, three ways to read it:")
print(pd.DataFrame(rows).to_string(index=False))

metrics = {"seed": SEED, "sklearn": sklearn.__version__, "eligible_pages": int(len(e)), "base_rate": round(float(y.mean()), 4),
           "before_after_precision_at_50": {k: {"Random Forest": round(float(res.loc[k, "Random Forest p@50"]), 3),
                                                "Logistic Regression": round(float(res.loc[k, "Logistic Regression p@50"]), 3),
                                                "Rule baseline": round(float(res.loc[k, "Rule baseline p@50"]), 3)} for k in res.index},
           "repeated_client_holdouts_20": {"LogReg_beats_rule": int(((rep['Logistic Regression'] - rep['rule']) > 0).sum()),
                                           "RF_beats_rule": int(((rep['Random Forest'] - rep['rule']) > 0).sum())},
           "comparison_articles_clients": int(len(comp_client))}
with open("work/outputs/w06_audit_metrics.json", "w") as f: json.dump(metrics, f, indent=2)
print("\nSaved work/outputs/w06_audit_metrics.json")

Clients that have each content type (whole file):
content_type
comparison article     1
feedly article         7
keyword article       31

Clients with comparison articles: 1 -> ['client_d029fa3a95']
Comparison articles at that client: 697 | its keyword articles: 255



Pooled CTR (%) comparison vs keyword, three ways to read it:
    tier  comparison_n  comparison_ctr_pct  kw_ALL_clients_n  kw_ALL_ctr_pct  kw_SAME_client_n  kw_SAME_ctr_pct      same_client_gap_95%_range
  page_1           226               0.077              8186           0.350                50            0.171                 0.034 to 0.161
striking            76               0.160              5752           0.347                32            0.160                -0.084 to 0.091
page_3_5            63               0.094              5954           0.155                 7            0.162 too few keyword pages (n < 30)

Saved work/outputs/w06_audit_metrics.json


### Why this section matters most

While checking my claims I found something that changes one of them. **Every comparison article in the file (697 of them) comes from a single client**, and that client's own keyword articles also get below-average click-through (0.17% on page 1 vs 0.35% for other clients' keyword articles). In Week 4 I called "comparison articles underperform" CONFIRMED by comparing them to *all* keyword articles. That comparison mixes content type with client, and I can't separate the two that way. The fairer test is within the one client that has both.

- **Page 1:** comparison 0.077% (226 pages) vs the same client's keyword 0.171% (50 pages). The bootstrap 95% range for the gap is 0.03 to 0.16 points, which excludes zero.
- **Striking distance:** 0.160% vs 0.160% (76 vs 32 pages). No gap; the range spans zero.
- **Pages 3-5:** only 7 keyword pages at that client, so I can't test it.

### My claims, rewritten

| Where | Before | After |
|---|---|---|
| Proof statement / Week 1 and Week 4 | "Comparison-style articles underperform across most ranking positions" (Week 4 verdict: CONFIRMED) | "At the one client that has them, comparison articles had lower click-through than that client's keyword articles on page 1 (0.08% vs 0.17%, on 226 vs 50 pages) but not in striking distance. All comparison articles in the file come from that client, so I can't separate content type from client." Verdict: **MIXED, confounded with client.** |
| Week 5 | "The model ranked pages about 10 points better than my rule" | "Measured on one snapshot of pages that stayed visible, a Logistic Regression beat my rule at precision@50 in 18 of 20 client holdouts, by about 0.11 on average (spread about 0.10). It is directional and decision-support only." |
| Week 5 | "The model mostly learns persistence of low CTR" | "The model leans on past clicks relative to impressions, in whatever form; removing the two CTR columns changed little (Random Forest 0.816 to 0.796)." |
| Week 5 | "Top-200 are 87.5% correct" | "87.5% against a 36% base rate, but from 13 clients with 44% from one, and only for pages with at least 500 impressions in the outcome window." |
| Scope | (not stated) | "41% of pages that were visible before the outcome window are excluded because they lost visibility. These results don't apply to them." |

### What I can and can't say

I can say the model's top-ranked pages were, on clients it hadn't seen, more often under-capturing in the next 30 days than my hand rule's top picks. I **cannot** say what causes low CTR, that rewriting titles will raise clicks, that comparison articles are weaker as a content type, or that this holds outside this snapshot. These are observed, directional numbers for decision support.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.